# Panel Judge — FULL DATASET RUN v4 (Patched)

Three-model panel (Claude Opus 4.7, GPT-5.5, Gemini 3.1 Pro) cross-judging hallucinations
on UCF-Crime model outputs.

## Patches applied

1. **Claude**: walks `resp.content` for the first text block instead of indexing `[0]`.
   Opus 4.7 may emit an empty thinking block at index 0 even with adaptive thinking off.
2. **Gemini safety**: thresholds set to `BLOCK_NONE` (UCF-Crime trips the violence filter).
   When `resp.text` is `None`, raises a clear error with `finish_reason` and `block_reason`.
3. **All judges: token budget bumped to 8192** *(v4)*. Gemini 3.1 Pro requires thinking
   mode and consumes ~975 tokens of internal thinking before writing JSON; GPT-5.5 also
   hides reasoning tokens in `max_completion_tokens`. Diagnostics on row 3480 (Fighting)
   showed `finish_reason=MAX_TOKENS` with the JSON cut off mid-key. 8192 leaves comfortable
   headroom for thinking + JSON across all three judges. The actual JSON answer is ~150
   tokens, so this only matters when reasoning runs long on hard prompts.
4. **JSON parser**: catches `json.JSONDecodeError` specifically (was being shadowed by
   `except ValueError`), so the regex fallback actually runs for malformed Gemini output.
5. **Cleanup**: preserves `PROHIBITED_CONTENT` errors — those are hard policy blocks
   that cannot succeed on retry, so we keep them in the checkpoint instead of looping forever.

## Version history

- **v1**: Claude block-walking + Gemini safety-off + parser regex fix.
- **v2**: Tried `thinking_budget=0` to disable Gemini thinking — rejected with 400.
- **v3**: Gemini budget 1024 → 4096 (kept thinking enabled).
- **v4**: All three judges bumped to 8192 after seeing GPT also truncate (row 3189: 6/6 missing).

## Run order

**Run cells top-to-bottom in order.** If you restart the kernel, you must re-run from
cell 2 (config) onward.

| Cell | Purpose |
|------|---------|
| 2    | Config: paths, API keys, constants |
| 4    | Auto-cleanup of bad checkpoint entries (preserves permanent failures) |
| 6    | Ground truth loader + Gemini extractors |
| 8    | Build/refresh triplets |
| 10   | SDK clients + prompt + retry helper + patched `parse_judge_response` |
| 12   | Judge functions: Claude / GPT / Gemini (all 8192 token budget) |
| 14   | Smoke test — verify all 3 judges return JSON before launching |
| 16   | Pre-flight: handles stale `PANEL_RAW_PATH` |
| 18   | **Main panel run** (resumes from checkpoint) |
| 20   | Inter-judge agreement (Cohen's Kappa) |
| 22   | Aggregate panel votes via majority |
| 24   | Final summary tables |


---
## 1. Configuration

In [1]:
import os
import json
import re
import sys
import time
import pandas as pd
import numpy as np
from collections import defaultdict, Counter
from tqdm.auto import tqdm

sys.stdout.flush()

# =====================================================================
# PATHS
# =====================================================================
RESULTS_BASE = r'C:\Opeyemi\PROMPTS\RESULTS'
GT_PATH      = r'C:\Opeyemi\PROMPTS\ANNOTATION'
OUTPUT_DIR   = r'C:\Opeyemi\PROMPTS\EVALUATION'
API_KEYS_DIR = r'C:\Opeyemi\PROMPTS\API-KEYS'

os.makedirs(OUTPUT_DIR, exist_ok=True)

BASE_PATHS = {
    'Claude': os.path.join(RESULTS_BASE, 'CLAUDE'),
    'GPT':    os.path.join(RESULTS_BASE, 'GPT'),
    'Gemini': os.path.join(RESULTS_BASE, 'GEMINI'),
}

# Cache files
GT_CACHE_PATH       = os.path.join(OUTPUT_DIR, 'gt_data_cache.json')
TRIPLETS_CACHE_PATH = os.path.join(OUTPUT_DIR, 'all_triplets_cache.csv')

# FULL run output files (separate from _subset files you already have)
PANEL_CHECKPOINT     = os.path.join(OUTPUT_DIR, 'panel_checkpoint_full.json')
PANEL_RAW_PATH       = os.path.join(OUTPUT_DIR, 'panel_raw_judge_labels_full.csv')
JUDGE_LABELS_PATH    = os.path.join(OUTPUT_DIR, 'llm_judge_labels_full.csv')
JUDGE_AGREEMENT_PATH = os.path.join(OUTPUT_DIR, 'inter_judge_agreement_full.json')
FULL_LABELED_PATH    = os.path.join(OUTPUT_DIR, 'full_labeled_dataset_full.csv')

# =====================================================================
# API KEYS
# =====================================================================
def load_key(filename):
    with open(os.path.join(API_KEYS_DIR, filename), 'r') as f:
        return f.read().strip()

ANTHROPIC_API_KEY = load_key('claude.txt')
OPENAI_API_KEY    = load_key('chatgpt.txt')
GEMINI_API_KEY    = load_key('ope-gemi.txt')

# Set Gemini env vars (matches the working ope-gemi pattern)
os.environ['GOOGLE_API_KEY'] = GEMINI_API_KEY
os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY

# =====================================================================
# CONSTANTS
# =====================================================================
TECHNIQUES = {
    'ZERO':          'Zero-Shot',
    'SEQUENTIAL':    'Sequential',
    'LEAST-TO-MOST': 'Least-to-Most',
    'REACT':         'ReAct',
}

ALL_JUDGES = ['Claude', 'GPT', 'Gemini']
JUDGE_FILTER = None   # None = all 3 judges (cross-judging, 2 per row)

def judges_for(model_name):
    cross = [j for j in ALL_JUDGES if j != model_name]
    if JUDGE_FILTER is not None:
        cross = [j for j in cross if j in JUDGE_FILTER]
    return cross

HTYPE_MAP = {
    'SCENE_FABRICATION':       'H1',
    'CRIME_MISCLASSIFICATION': 'H2',
    'CRIME_MISSED':            'H3',
    'SEVERITY_MINIMIZATION':   'H4',
    'ENTITY_FABRICATION':      'H5',
    'PHANTOM_ACTORS':          'H6',
}

# =====================================================================
# SANITY CHECKS
# =====================================================================
assert os.path.isdir(RESULTS_BASE), f'Missing: {RESULTS_BASE}'
assert os.path.isdir(GT_PATH),      f'Missing: {GT_PATH}'
assert os.path.isdir(API_KEYS_DIR), f'Missing: {API_KEYS_DIR}'
assert os.environ.get('GOOGLE_API_KEY'), 'GOOGLE_API_KEY not set'

print('=' * 70)
print('FULL PANEL RUN — Configuration')
print('=' * 70)
print(f'Results base:  {RESULTS_BASE}')
print(f'Output dir:    {OUTPUT_DIR}')
print(f'Anthropic key: {"loaded" if ANTHROPIC_API_KEY else "MISSING"}')
print(f'OpenAI key:    {"loaded" if OPENAI_API_KEY else "MISSING"}')
print(f'Gemini key:    {"loaded" if GEMINI_API_KEY else "MISSING"} (ope-gemi.txt, Developer API)')
print(f'Panel:         {ALL_JUDGES}')
print()
print(f'Output files (will be created):')
print(f'  {PANEL_CHECKPOINT}')
print(f'  {PANEL_RAW_PATH}')
print(f'  {JUDGE_LABELS_PATH}')
print(f'  {JUDGE_AGREEMENT_PATH}')
print(f'  {FULL_LABELED_PATH}')

FULL PANEL RUN — Configuration
Results base:  C:\Opeyemi\PROMPTS\RESULTS
Output dir:    C:\Opeyemi\PROMPTS\EVALUATION
Anthropic key: loaded
OpenAI key:    loaded
Gemini key:    loaded (ope-gemi.txt, Developer API)
Panel:         ['Claude', 'GPT', 'Gemini']

Output files (will be created):
  C:\Opeyemi\PROMPTS\EVALUATION\panel_checkpoint_full.json
  C:\Opeyemi\PROMPTS\EVALUATION\panel_raw_judge_labels_full.csv
  C:\Opeyemi\PROMPTS\EVALUATION\llm_judge_labels_full.csv
  C:\Opeyemi\PROMPTS\EVALUATION\inter_judge_agreement_full.json
  C:\Opeyemi\PROMPTS\EVALUATION\full_labeled_dataset_full.csv


---
## 2. Auto-clean retryable errors (preserve permanent ones)

In [2]:
# Auto-cleanup of bad checkpoint entries.
# Permanent policy blocks (Gemini PROHIBITED_CONTENT) are preserved — retrying them
# is futile, and we want the run to terminate. Other -1 entries are removed so they
# get retried with the patched judge functions.

if os.path.exists(PANEL_CHECKPOINT):
    with open(PANEL_CHECKPOINT) as f:
        _ckpt = json.load(f)
    _initial = len(_ckpt)

    _bad_keys = []
    _kept_permanent = 0
    for k, v in _ckpt.items():
        if not isinstance(v, dict):
            continue
        has_neg1 = any(v.get(h, 0) == -1 for h in ['H1','H2','H3','H4','H5','H6'])
        if not has_neg1:
            continue
        reasoning = str(v.get('reasoning', ''))
        # Don't retry hard policy blocks — they will never succeed
        if 'PROHIBITED_CONTENT' in reasoning:
            _kept_permanent += 1
            continue
        _bad_keys.append(k)

    for k in _bad_keys:
        del _ckpt[k]

    if _bad_keys:
        with open(PANEL_CHECKPOINT, 'w') as f:
            json.dump(_ckpt, f)
        print(f'Auto-cleanup: removed {len(_bad_keys)} retryable errors')
        print(f'  Checkpoint: {_initial} -> {len(_ckpt)} entries')
    else:
        print(f'Auto-cleanup: nothing retryable to clean ({_initial} entries)')
    if _kept_permanent:
        print(f'  Preserved {_kept_permanent} permanent policy-block entries (will not retry)')
else:
    print('No existing full checkpoint - starting fresh.')

Auto-cleanup: removed 1 retryable errors
  Checkpoint: 7075 -> 7074 entries
  Preserved 1 permanent policy-block entries (will not retry)


---
## 3. Ground truth

In [3]:
# Ground truth loader
GT_FILES = ['UCFCrime_Train.json', 'UCFCrime_Val.json', 'UCFCrime_Test.json']

def derive_crime_type(video_id):
    name = video_id.replace('_x264', '')
    m = re.match(r'^([A-Za-z]+?)\d', name)
    return m.group(1) if m else 'Unknown'

def is_anomalous(video_id):
    return not video_id.startswith('Normal_')

def parse_ground_truth(gt_base_path):
    gt_data = {}
    for fname in GT_FILES:
        fpath = os.path.join(gt_base_path, fname)
        if not os.path.exists(fpath):
            continue
        with open(fpath, 'r', encoding='utf-8') as f:
            data = json.load(f)
        for video_id, info in data.items():
            if not is_anomalous(video_id):
                continue
            gt_data[video_id] = {
                'crime_type': derive_crime_type(video_id),
                'sentences':  info.get('sentences', []),
                'timestamps': info.get('timestamps', []),
                'duration':   info.get('duration'),
            }
    return gt_data

if os.path.exists(GT_CACHE_PATH):
    with open(GT_CACHE_PATH) as f:
        gt_data = json.load(f)
    print(f'Loaded GT from cache: {len(gt_data):,} videos')
else:
    gt_data = parse_ground_truth(GT_PATH)
    with open(GT_CACHE_PATH, 'w') as f:
        json.dump(gt_data, f)
    print(f'Cached {len(gt_data):,} GT videos')

# Video ID helpers
GEMINI_VID_RE = re.compile(r'^([A-Za-z]+_[A-Za-z]+\d+(?:_x264)?)')

def video_id_from_filename(filename):
    m = GEMINI_VID_RE.match(filename)
    return m.group(1) if m else None

def resolve_gt_key(video_id, gt_data):
    candidates = [video_id, video_id.replace('_x264', '')]
    parts = video_id.split('_')
    if len(parts) >= 2:
        wp = '_'.join(parts[1:])
        candidates += [wp, wp.replace('_x264', '')]
    for c in candidates:
        if c in gt_data:
            return c
    return None

# Gemini extractors (only used by Stage 1 to refresh new Gemini videos)
def extract_gemini_sequential_obj(obj):
    results = obj.get('sequential_results', obj)
    if not isinstance(results, dict): return ''
    final = results.get('Final Synthesis', {})
    if isinstance(final, dict):
        text = final.get('response', '')
        if text and text.strip(): return text.strip()
    if isinstance(final, str) and final.strip(): return final.strip()
    step_keys = sorted(
        [k for k in results if k.startswith('Step')],
        key=lambda x: int(x.split()[-1]) if x.split()[-1].isdigit() else 0,
        reverse=True
    )
    if step_keys:
        val = results[step_keys[0]]
        if isinstance(val, dict): return val.get('response', '')
        return str(val)
    return ''

def extract_gemini_ltm_obj(obj):
    results = obj.get('least_to_most_results', obj)
    if not isinstance(results, dict): return ''
    step_keys = sorted(
        [k for k in results if k.startswith('Step')],
        key=lambda x: int(x.split()[-1]) if x.split()[-1].isdigit() else 0,
        reverse=True
    )
    if not step_keys: return ''
    last = results[step_keys[0]]
    if isinstance(last, dict): return last.get('response', '')
    return last if isinstance(last, str) else ''

def load_gemini_running_tech(tech_folder):
    """Read all currently-completed Gemini videos for one in-progress technique."""
    tech_path = os.path.join(BASE_PATHS['Gemini'], tech_folder)
    if not os.path.isdir(tech_path):
        return {}
    extractor = (extract_gemini_sequential_obj if tech_folder == 'SEQUENTIAL'
                 else extract_gemini_ltm_obj)
    all_files = sorted([f for f in os.listdir(tech_path)
                        if f.endswith('.json') and '_checkpoints' not in f])
    complete  = [f for f in all_files if '_complete_' in f]
    per_video = defaultdict(list)
    for fname in complete:
        try:
            with open(os.path.join(tech_path, fname), encoding='utf-8') as f:
                obj = json.load(f)
        except Exception:
            continue
        text = extractor(obj)
        vid = video_id_from_filename(fname)
        if vid and text and text.strip():
            per_video[vid].append(text.strip())
    return {vid: texts[-1] for vid, texts in per_video.items()}

print('Helpers ready.')

Loaded GT from cache: 944 videos
Helpers ready.


---
## 4. Build / refresh triplets

In [4]:
if not os.path.exists(TRIPLETS_CACHE_PATH):
    raise FileNotFoundError(
        f'{TRIPLETS_CACHE_PATH} missing. Run the original build_triplets cell first.'
    )

df_old = pd.read_csv(TRIPLETS_CACHE_PATH)
print(f'Existing cache: {len(df_old):,} rows')
print('  Per (model, technique):')
print(df_old.groupby(['model', 'technique']).size().unstack(fill_value=0))

existing_keys = set(zip(df_old['model'], df_old['technique'], df_old['video']))

RUNNING_TECHS = {
    'SEQUENTIAL':    'Sequential',
    'LEAST-TO-MOST': 'Least-to-Most',
}

new_rows = []
for tech_folder, tech_label in RUNNING_TECHS.items():
    outputs = load_gemini_running_tech(tech_folder)
    on_disk = len(outputs)
    added = 0
    for video_id, output_text in outputs.items():
        gt_key = resolve_gt_key(video_id, gt_data)
        if gt_key is None:
            continue
        key = ('Gemini', tech_label, gt_key)
        if key in existing_keys:
            continue
        gt_info = gt_data[gt_key]
        new_rows.append({
            'model':                 'Gemini',
            'technique':             tech_label,
            'video':                 gt_key,
            'crime_type':            gt_info['crime_type'],
            'ground_truth':          ' '.join(gt_info['sentences']),
            'model_output':          output_text,
            'model_output_full_len': len(output_text),
        })
        added += 1
    print(f'  [Gemini/{tech_label}] on disk: {on_disk}  NEW added: {added}')

if new_rows:
    df_new = pd.DataFrame(new_rows)
    df_triplets = pd.concat([df_old, df_new], ignore_index=True)
    df_triplets.to_csv(TRIPLETS_CACHE_PATH, index=False)
    print(f'\nCache: {len(df_old):,} -> {len(df_triplets):,}  (+{len(new_rows)} new)')
else:
    df_triplets = df_old
    print('\nNo new Gemini videos found — cache already up to date.')

print('\nFinal triplet pool per (model, technique):')
print(df_triplets.groupby(['model', 'technique']).size().unstack(fill_value=0))
print(f'\nTotal rows: {len(df_triplets):,}')
print(f'Total judge calls if fresh: {len(df_triplets) * 2:,}')

Existing cache: 9,680 rows
  Per (model, technique):
technique  Least-to-Most  ReAct  Sequential  Zero-Shot
model                                                 
Claude               807    807         804        807
GPT                  807    807         807        807
Gemini               806    807         807        807
  [Gemini/Sequential] on disk: 812  NEW added: 0
  [Gemini/Least-to-Most] on disk: 811  NEW added: 0

No new Gemini videos found — cache already up to date.

Final triplet pool per (model, technique):
technique  Least-to-Most  ReAct  Sequential  Zero-Shot
model                                                 
Claude               807    807         804        807
GPT                  807    807         807        807
Gemini               806    807         807        807

Total rows: 9,680
Total judge calls if fresh: 19,360


---
## 5. SDK clients + prompt + retry helpers

In [5]:
import anthropic
import openai
from google import genai as google_genai
from google.genai import types as google_genai_types

_anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
_openai_client    = openai.OpenAI(api_key=OPENAI_API_KEY)
# Gemini Developer API (not Vertex AI) - plain API key
_genai_client     = google_genai.Client(
    api_key=os.environ['GOOGLE_API_KEY'],
)

JUDGE_SYSTEM = """You are an expert forensic video analysis evaluator.
Your task is to detect hallucinations in an AI model's analysis of a crime video.
You will be given:
  1. GROUND TRUTH: the human-annotated description of what actually happened.
  2. MODEL OUTPUT: the AI system's analysis of the same video.

Evaluate whether the MODEL OUTPUT contains any of the following hallucination types.
For each type respond ONLY with 0 (not present) or 1 (present).

Hallucination types:
  H1 SCENE_FABRICATION    - invents setting details not in ground truth (wrong location, objects, environment)
  H2 CRIME_MISCLASSIFICATION - identifies the wrong crime type (e.g. says Robbery when it is Assault)
  H3 CRIME_MISSED         - fails to mention the primary crime that is clearly described in ground truth
  H4 SEVERITY_MINIMIZATION - describes crime as less serious than ground truth indicates
  H5 ENTITY_FABRICATION   - invents people, vehicles, or objects not present in ground truth
  H6 PHANTOM_ACTORS       - adds perpetrators or victims not mentioned in ground truth

Respond ONLY with a JSON object in this exact format, no extra text:
{
  "H1": 0,
  "H2": 0,
  "H3": 0,
  "H4": 0,
  "H5": 0,
  "H6": 0,
  "reasoning": "brief one-sentence explanation"
}"""

def make_judge_prompt(ground_truth: str, model_output: str) -> str:
    gt_trunc  = ground_truth[:1500]
    out_trunc = model_output[:3000]
    return (
        f"GROUND TRUTH:\n{gt_trunc}\n\n"
        f"MODEL OUTPUT:\n{out_trunc}\n\n"
        "Now evaluate for hallucinations and return the JSON."
    )

def _retry_call(fn, max_retries=7, base_delay=4.0, max_delay=300.0):
    last_err = None
    for attempt in range(max_retries):
        try:
            return fn()
        except Exception as e:
            last_err = e
            err_str  = str(e)
            err_low  = err_str.lower()
            is_rate    = any(x in err_low for x in ['429', 'rate limit', 'quota', 'resource exhausted', 'resource_exhausted'])
            is_server  = any(x in err_low for x in ['500', '502', '503', '504', 'server error', 'unavailable', 'overloaded'])
            is_timeout = any(x in err_low for x in ['timeout', 'timed out', 'deadline'])
            is_truncated = 'truncated response' in err_low
            retryable = is_rate or is_server or is_timeout or is_truncated
            if retryable and attempt < max_retries - 1:
                delay = min(base_delay * (2 ** attempt), max_delay)
                short_err = err_str[:250].replace('\n', ' ')
                tqdm.write(f'    [{type(e).__name__} attempt {attempt+1}/{max_retries}] sleeping {delay:.0f}s | {short_err}')
                time.sleep(delay)
            else:
                raise
    if last_err:
        raise last_err
    return None

def parse_judge_response(text: str) -> dict:
    """Parse a judge JSON response, with regex fallback for malformed output.

    Patched: catches json.JSONDecodeError specifically (not all ValueErrors)
    so the regex fallback actually runs for malformed JSON like single-quoted
    keys, trailing commas, or unescaped quotes in the reasoning field.
    """
    text = re.sub(r'```(?:json)?', '', text).strip().rstrip('`').strip()

    # Try strict JSON first
    obj = None
    try:
        obj = json.loads(text)
    except json.JSONDecodeError:
        pass  # fall through to regex fallback

    if obj is not None:
        labels = {}
        for h in ['H1','H2','H3','H4','H5','H6']:
            val = obj.get(h, -1)
            labels[h] = int(bool(val)) if val in (0, 1, True, False) else -1
        labels['reasoning'] = str(obj.get('reasoning', ''))
        n_missing = sum(1 for h in ['H1','H2','H3','H4','H5','H6'] if labels[h] == -1)
        if n_missing > 0:
            raise ValueError(f'Truncated response: {n_missing} of 6 fields missing')
        return labels

    # Regex fallback for malformed JSON (single quotes, trailing commas, etc.)
    labels = {}
    for h in ['H1','H2','H3','H4','H5','H6']:
        m = re.search(r'["\']?' + h + r'["\']?\s*:\s*([01])', text)
        labels[h] = int(m.group(1)) if m else -1
    rm = re.search(r'["\']?reasoning["\']?\s*:\s*["\']([^"\']+)["\']', text)
    labels['reasoning'] = rm.group(1) if rm else 'parsed via regex fallback'
    n_missing = sum(1 for h in ['H1','H2','H3','H4','H5','H6'] if labels[h] == -1)
    if n_missing > 0:
        raise ValueError(f'Truncated response: {n_missing} of 6 fields missing')
    return labels

print('Prompt + retry helper + patched parse_judge_response ready.')


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Prompt + retry helper + patched parse_judge_response ready.


---
## 6. Judge functions (patched)

In [6]:
def _extract_text_from_anthropic_response(resp):
    """Find the text block in an Anthropic response, skipping any thinking blocks.

    On Claude Opus 4.7, the response stream can include an empty thinking block
    as content[0] even when adaptive thinking is off (display defaults to "omitted").
    The actual answer text is in a later block with type == "text".
    """
    for block in resp.content:
        if getattr(block, 'type', None) == 'text':
            txt = getattr(block, 'text', None)
            if txt:
                return txt
    parts = []
    for block in resp.content:
        t = getattr(block, 'text', None)
        if t:
            parts.append(t)
    if parts:
        return '\n'.join(parts)
    raise ValueError(
        f'No text block in Anthropic response. '
        f'Blocks: {[getattr(b, "type", "?") for b in resp.content]}'
    )


def call_claude_judge(ground_truth: str, model_output: str) -> dict:
    prompt = make_judge_prompt(ground_truth, model_output)
    def _call():
        resp = _anthropic_client.messages.create(
            model='claude-opus-4-7',
            max_tokens=8192,
            system=JUDGE_SYSTEM,
            messages=[{'role': 'user', 'content': prompt}],
            # NOTE: no temperature - Opus 4.7 returns 400 on non-default temp/top_p/top_k
        )
        return _extract_text_from_anthropic_response(resp)
    raw = _retry_call(_call)
    return parse_judge_response(raw)


def call_gpt_judge(ground_truth: str, model_output: str) -> dict:
    prompt = make_judge_prompt(ground_truth, model_output)
    def _call():
        resp = _openai_client.chat.completions.create(
            model='gpt-5.5',
            messages=[
                {'role': 'system', 'content': JUDGE_SYSTEM},
                {'role': 'user',   'content': prompt},
            ],
            max_completion_tokens=8192,
        )
        return resp.choices[0].message.content
    raw = _retry_call(_call)
    return parse_judge_response(raw)


def call_gemini_judge(ground_truth: str, model_output: str) -> dict:
    prompt = make_judge_prompt(ground_truth, model_output)
    full_prompt = JUDGE_SYSTEM + '\n\n' + prompt

    # Disable output safety filters - UCF-Crime descriptions trip BLOCK_MEDIUM and
    # cause resp.text to return None. We're evaluating, not generating new content.
    # Note: the input PROHIBITED_CONTENT filter is a hard policy block that cannot
    # be disabled at any threshold; those failures are permanent and surfaced below.
    _safety = [
        google_genai_types.SafetySetting(category=c, threshold='BLOCK_NONE')
        for c in (
            'HARM_CATEGORY_HARASSMENT',
            'HARM_CATEGORY_HATE_SPEECH',
            'HARM_CATEGORY_SEXUALLY_EXPLICIT',
            'HARM_CATEGORY_DANGEROUS_CONTENT',
        )
    ]
    _gemini_config = google_genai_types.GenerateContentConfig(
        temperature=0.1,
        # Gemini 3.1 Pro requires thinking mode; we can't disable it. Instead, we give
        # enough total output budget for thinking + JSON to coexist. Diagnostics showed
        # thinking consumed ~975 tokens on Fighting prompts, leaving the 1024 budget
        # cut off mid-JSON. 4096 leaves comfortable room for both.
        max_output_tokens=8192,
        top_p=0.8,
        top_k=10,
        safety_settings=_safety,
    )

    def _call():
        try:
            resp = _genai_client.models.generate_content(
                model='gemini-3.1-pro-preview',
                contents=full_prompt,
                config=_gemini_config,
            )
        except Exception as e:
            err = str(e)
            if '429' in err or 'RESOURCE_EXHAUSTED' in err or 'quota' in err.lower():
                tqdm.write('    [Gemini 429] sleeping 60s before retry...')
                time.sleep(60)
            raise

        # resp.text is None when blocked / no candidates / no parts.
        # Surface a clear reason instead of letting parse_judge_response choke on None.
        text = resp.text
        if text is None:
            finish = None
            block_reason = None
            try:
                if resp.candidates:
                    finish = getattr(resp.candidates[0], 'finish_reason', None)
                if getattr(resp, 'prompt_feedback', None):
                    block_reason = getattr(resp.prompt_feedback, 'block_reason', None)
            except Exception:
                pass
            raise ValueError(
                f'Gemini returned no text (finish_reason={finish}, block_reason={block_reason})'
            )
        return text

    raw = _retry_call(_call)
    return parse_judge_response(raw)


JUDGE_CALL_MAP = {
    'Claude': call_claude_judge,
    'GPT':    call_gpt_judge,
    'Gemini': call_gemini_judge,
}

print('Judge functions ready.')

Judge functions ready.


---
## 7. Smoke test

Run before launching the full panel to confirm both Claude and Gemini are returning valid JSON.

In [7]:
test_gt  = "A person enters a store and takes items without paying."
test_out = "The video shows shoplifting at a convenience store."

print('Claude:', call_claude_judge(test_gt, test_out))
print('GPT:   ', call_gpt_judge(test_gt, test_out))
print('Gemini:', call_gemini_judge(test_gt, test_out))

Claude: {'H1': 0, 'H2': 0, 'H3': 0, 'H4': 0, 'H5': 0, 'H6': 0, 'reasoning': "Model correctly identifies shoplifting; 'convenience store' is a reasonable specification consistent with ground truth."}
GPT:    {'H1': 1, 'H2': 0, 'H3': 0, 'H4': 0, 'H5': 0, 'H6': 0, 'reasoning': 'The model correctly identifies shoplifting but adds the unsupported detail that it occurred at a convenience store.'}
Gemini: {'H1': 1, 'H2': 0, 'H3': 0, 'H4': 0, 'H5': 0, 'H6': 0, 'reasoning': "The model fabricates the specific type of store ('convenience store') which is not detailed in the ground truth."}


---
## 8. Pre-flight check

If `PANEL_RAW_PATH` already exists, the panel cell will skip the run and just load the stale CSV. Rename it out of the way before launching.

In [8]:
print('PANEL_RAW_PATH exists:', os.path.exists(PANEL_RAW_PATH))
print('PANEL_CHECKPOINT exists:', os.path.exists(PANEL_CHECKPOINT))

if os.path.exists(PANEL_RAW_PATH):
    backup = PANEL_RAW_PATH + '.stale'
    os.rename(PANEL_RAW_PATH, backup)
    print(f'Renamed stale CSV out of the way -> {backup}')

PANEL_RAW_PATH exists: False
PANEL_CHECKPOINT exists: True


---
## 9. Run the panel

In [9]:
def load_checkpoint():
    if os.path.exists(PANEL_CHECKPOINT):
        with open(PANEL_CHECKPOINT) as f:
            return json.load(f)
    return {}

def save_checkpoint(ckpt):
    with open(PANEL_CHECKPOINT, 'w') as f:
        json.dump(ckpt, f)

def run_judge_panel(df, max_workers=8):
    from concurrent.futures import ThreadPoolExecutor, as_completed
    import threading

    ckpt = load_checkpoint()
    ckpt_lock = threading.Lock()
    rows_lock = threading.Lock()
    rows_out = []
    tasks = []
    cached_rows = []
    h_keys = ['H1','H2','H3','H4','H5','H6']

    for idx, row in df.iterrows():
        model      = row['model']
        technique  = row['technique']
        video      = row['video']
        crime_type = row['crime_type']
        gt         = str(row['ground_truth'])
        output     = str(row['model_output'])
        for judge in judges_for(model):
            ck_key = f'{idx}_{judge}'
            base_row = {
                'row_idx':    idx,
                'model':      model,
                'technique':  technique,
                'video':      video,
                'crime_type': crime_type,
                'judge':      judge,
            }
            if ck_key in ckpt:
                labels = ckpt[ck_key]
                cached_rows.append({
                    **base_row,
                    **{h: labels.get(h, -1) for h in h_keys},
                    'reasoning': labels.get('reasoning', ''),
                })
            else:
                tasks.append((idx, base_row, judge, gt, output, ck_key))

    n_judges = len(JUDGE_FILTER) if JUDGE_FILTER else 2
    print(f'\nPanel: {len(df):,} triplets × {n_judges} judges = {len(tasks)+len(cached_rows):,} calls')
    print(f'Already cached: {len(cached_rows):,}')
    print(f'Remaining:      {len(tasks):,}')
    print(f'Workers:        {max_workers}')

    rows_out.extend(cached_rows)
    if not tasks:
        print('Nothing to do.')
        return pd.DataFrame(rows_out)

    pbar = tqdm(total=len(tasks), desc='Judge calls', unit='call', dynamic_ncols=True)

    def _process_task(task):
        idx, base_row, judge, gt, output, ck_key = task
        try:
            labels = JUDGE_CALL_MAP[judge](gt, output)
        except Exception as e:
            tqdm.write(f'  ERROR [{judge}] row {idx}: {str(e)[:200]}')
            labels = {h: -1 for h in h_keys}
            labels['reasoning'] = f'ERROR: {str(e)[:300]}'
        row_out = {
            **base_row,
            **{h: labels.get(h, -1) for h in h_keys},
            'reasoning': labels.get('reasoning', ''),
        }
        with ckpt_lock:
            ckpt[ck_key] = labels
            if len(ckpt) % 25 == 0:
                save_checkpoint(ckpt)
        with rows_lock:
            rows_out.append(row_out)
        return row_out

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = [ex.submit(_process_task, t) for t in tasks]
        for _ in as_completed(futures):
            pbar.update(1)
    pbar.close()

    with ckpt_lock:
        save_checkpoint(ckpt)
    return pd.DataFrame(rows_out)

# EXECUTE PANEL
if os.path.exists(PANEL_RAW_PATH):
    df_panel_raw = pd.read_csv(PANEL_RAW_PATH)
    print(f'Loaded existing raw labels: {len(df_panel_raw):,} rows')
else:
    df_panel_raw = run_judge_panel(df_triplets, max_workers=8)
    df_panel_raw.to_csv(PANEL_RAW_PATH, index=False)
    print(f'\nSaved raw labels: {len(df_panel_raw):,} rows')

df_panel_raw.to_csv(JUDGE_LABELS_PATH, index=False)


Panel: 9,680 triplets × 2 judges = 19,360 calls
Already cached: 7,100
Remaining:      12,260
Workers:        8


Judge calls:   0%|          | 0/12260 [00:00<?, ?call/s]

    [ServerError attempt 1/7] sleeping 4s | 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The service is currently unavailable.', 'status': 'UNAVAILABLE'}}
  ERROR [Gemini] row 5682: Gemini returned no text (finish_reason=None, block_reason=PROHIBITED_CONTENT)

Saved raw labels: 19,360 rows


---
## 10. Inter-judge agreement

In [10]:
def cohens_kappa(y1, y2):
    valid = [(a, b) for a, b in zip(y1, y2) if a in (0, 1) and b in (0, 1)]
    if len(valid) < 2:
        return float('nan')
    a = [v[0] for v in valid]
    b = [v[1] for v in valid]
    n = len(a)
    p_o = sum(x == y for x, y in zip(a, b)) / n
    p_a = (sum(a) / n) * (sum(b) / n) + ((n - sum(a)) / n) * ((n - sum(b)) / n)
    return (p_o - p_a) / (1 - p_a) if p_a < 1 else 1.0

def compute_inter_judge_agreement(df_raw):
    h_cols = ['H1','H2','H3','H4','H5','H6']
    judges = df_raw['judge'].unique().tolist()
    pairs = [(judges[i], judges[j])
             for i in range(len(judges)) for j in range(i+1, len(judges))]
    results = {}
    for j1, j2 in pairs:
        pair_key = f'{j1}_vs_{j2}'
        results[pair_key] = {}
        df1 = df_raw[df_raw['judge'] == j1].set_index('row_idx')
        df2 = df_raw[df_raw['judge'] == j2].set_index('row_idx')
        common_idx = df1.index.intersection(df2.index)
        for h in h_cols:
            y1 = df1.loc[common_idx, h].tolist()
            y2 = df2.loc[common_idx, h].tolist()
            results[pair_key][h] = round(cohens_kappa(y1, y2), 4)
        y1_all = [v for h in h_cols for v in df1.loc[common_idx, h].tolist()]
        y2_all = [v for h in h_cols for v in df2.loc[common_idx, h].tolist()]
        results[pair_key]['overall'] = round(cohens_kappa(y1_all, y2_all), 4)
    return results

print('Computing inter-judge agreement...')
agreement = compute_inter_judge_agreement(df_panel_raw)
with open(JUDGE_AGREEMENT_PATH, 'w') as f:
    json.dump(agreement, f, indent=2)

print('\nInter-judge agreement (Cohen\'s Kappa):')
for pair, kappas in agreement.items():
    print(f'  {pair}:')
    for h, k in kappas.items():
        print(f'    {h}: {k:.4f}')

Computing inter-judge agreement...

Inter-judge agreement (Cohen's Kappa):
  GPT_vs_Gemini:
    H1: 0.7745
    H2: 0.7348
    H3: 0.8160
    H4: 0.6053
    H5: 0.7464
    H6: 0.7063
    overall: 0.8166
  GPT_vs_Claude:
    H1: 0.1056
    H2: 0.5649
    H3: 0.6581
    H4: 0.6244
    H5: 0.0875
    H6: 0.5322
    overall: 0.5022
  Gemini_vs_Claude:
    H1: 0.2311
    H2: 0.6907
    H3: 0.7906
    H4: 0.4835
    H5: 0.3434
    H6: 0.4885
    overall: 0.5335


---
## 11. Aggregate panel votes

In [11]:
def majority_vote(vals):
    valid = [v for v in vals if v in (0, 1)]
    if not valid:
        return -1
    return 1 if sum(valid) >= len(valid) / 2 else 0

def aggregate_panel(df_raw, df_triplets):
    h_cols = ['H1','H2','H3','H4','H5','H6']
    agg = (df_raw
           .groupby(['row_idx','model','technique','video','crime_type'])[h_cols]
           .agg(majority_vote)
           .reset_index())
    inv_map = {v: k for k, v in HTYPE_MAP.items()}
    for h in h_cols:
        agg[inv_map[h]] = agg[h]
    df_t = df_triplets.reset_index().rename(columns={'index': 'row_idx'})
    agg = agg.merge(
        df_t[['row_idx','ground_truth','model_output','model_output_full_len']],
        on='row_idx', how='left'
    )
    agg['hallucination_count'] = agg[h_cols].apply(
        lambda r: sum(v for v in r if v == 1), axis=1
    )
    agg['any_hallucination'] = (agg['hallucination_count'] > 0).astype(int)
    return agg

print('Aggregating votes...')
df_full = aggregate_panel(df_panel_raw, df_triplets)
df_full.to_csv(FULL_LABELED_PATH, index=False)
print(f'Saved: {FULL_LABELED_PATH}  ({len(df_full):,} rows)')

Aggregating votes...
Saved: C:\Opeyemi\PROMPTS\EVALUATION\full_labeled_dataset_full.csv  (9,680 rows)


---
## 12. Summary

In [12]:
h_cols = ['H1','H2','H3','H4','H5','H6']
h_names = {
    'H1': 'Scene Fabrication',
    'H2': 'Crime Misclassification',
    'H3': 'Crime Missed',
    'H4': 'Severity Minimization',
    'H5': 'Entity Fabrication',
    'H6': 'Phantom Actors',
}

valid_df = df_full[df_full[h_cols].apply(lambda r: all(v >= 0 for v in r), axis=1)]

print('=' * 70)
print('FULL DATASET — Hallucination Summary')
print('=' * 70)
print(f'\nValid rows: {len(valid_df):,} of {len(df_full):,}')

print(f'\nHallucination rate by type:')
for h in h_cols:
    rate = valid_df[h].mean() * 100
    print(f'  {h} {h_names[h]:<26} {rate:5.1f}%')

print(f'\nOverall hallucination rate by model:')
print(valid_df.groupby('model')['any_hallucination'].mean().mul(100).round(1).to_string())

print(f'\nOverall hallucination rate by technique:')
print(valid_df.groupby('technique')['any_hallucination'].mean().mul(100).round(1).to_string())

print(f'\nHallucination rate by (model × technique):')
pivot = (valid_df.groupby(['model', 'technique'])['any_hallucination']
         .mean().mul(100).round(1).unstack(fill_value=0))
print(pivot)

print('\n' + '=' * 70)
print('Done. All _full output files saved to:', OUTPUT_DIR)
print('=' * 70)

FULL DATASET — Hallucination Summary

Valid rows: 9,680 of 9,680

Hallucination rate by type:
  H1 Scene Fabrication           85.4%
  H2 Crime Misclassification     30.3%
  H3 Crime Missed                51.5%
  H4 Severity Minimization       43.7%
  H5 Entity Fabrication          84.9%
  H6 Phantom Actors              38.6%

Overall hallucination rate by model:
model
Claude    99.3
GPT       96.8
Gemini    99.7

Overall hallucination rate by technique:
technique
Least-to-Most    99.0
ReAct            99.4
Sequential       99.2
Zero-Shot        96.7

Hallucination rate by (model × technique):
technique  Least-to-Most  ReAct  Sequential  Zero-Shot
model                                                 
Claude              98.0  100.0        99.1      100.0
GPT                 99.4   98.6        98.8       90.3
Gemini              99.5   99.5        99.8       99.9

Done. All _full output files saved to: C:\Opeyemi\PROMPTS\EVALUATION
